# A06: Patrones de Diseño en Python

**Nivel:** Avanzado | **Duración estimada:** 90-120 min

> Los patrones de diseño son **recetas probadas** para resolver problemas recurrentes de arquitectura de software. Aunque nacieron en los 90 con lenguajes como C++ y Java, en Python muchos de ellos se *simplifican radicalmente* gracias a las features del lenguaje.

---

## Tabla de Contenidos

1. ¿Qué son los patrones GoF y por qué en Python cambian?
2. Patrones Creacionales
3. Patrones Estructurales
4. Patrones Comportamentales
5. Patrones que en Python se vuelven "nativos"
6. Anti-patrones a evitar
7. Tabla de referencia
8. Ejercicios
9. Resumen

---

## Objetivos

Al finalizar este notebook serás capaz de:

1. **Explicar** qué son los 23 patrones GoF y por qué en Python muchos se vuelven idiomáticos (más simples).
2. **Aplicar** los patrones creacionales clave: Factory Method, Abstract Factory, Singleton y Builder.
3. **Aplicar** los patrones estructurales: Adapter, Facade, Proxy y Decorator.
4. **Aplicar** los patrones comportamentales: Strategy, Observer, Command y Template Method.
5. **Identificar** anti-patrones comunes y saber cuándo *no* usar un patrón (evitar sobre-ingeniería).
6. **Diseñar** un sistema completo combinando Observer + Strategy + Factory usando idiomas propios de Python.

---

## Analogía

Imagina que eres chef y tienes **recetas probadas**:

```
Problema recurrente        Receta (patrón)        Variación local
-----------------------------------------------------------------------
"Hacer pan"        -->    "Receta de pan"  --> "Pan integral / de maíz"
"Crear objetos"    -->    "Factory"       -->  "DB / API / archivo"
"Notificar muchos" -->    "Observer"      -->  "Email / SMS / push"
```

El patrón **no** te da código listo: te da la *estructura
tuya/ contexto específico. Es la misma idea. En **Python la receta se vuelve mucho más corta** porque el lenguaje ya trae muchos "ingredientes verdes":

- **Duck typing**: no necesitas interfaces formales, solo "que camine como pato".
- **Funciones de primera clase**: puedes pasar estrategias como simples funciones, sin clases.
- **Decoradores**: envuelven comportamiento de forma declarativa.
- **Protocolos / clases ABC**: abstracción ligera cuando la necesitas.

Esto hace que **se aprecie el problema** (cuándo aplicar cada patrón) más que la maquinaria del patrón (que en Python suele desaparecer).

---

## 1. ¿Qué son los patrones GoF y por qué en Python cambian?

### Historia

En 1994 el *Gang of Four* (Erich Gamma, Richard Helm, Ralph Johnson, John Vlissides) publicó *Design Patterns: Elements of Reusable Object-Oriented Software*, con **23 patrones** divididos en tres familias:

| Familia | Propósito | Ejemplos |
|---------|-----------|----------|
| **Creacionales** | Cómo se crean los objetos | Factory, Singleton, Builder, Abstract Factory |
| **Estructurales** | Cómo se componen las clases/objetos | Adapter, Facade, Proxy, Decorator |
| **Comportamentales** | Cómo se distribuyen responsabilidades | Strategy, Observer, Command, Template Method |

### ¿Por qué cambian en Python?

Muchos patrones resolvían **limitaciones de los lenguajes estáticos** (C++/Java) que Python no tiene:

```
Limitacion de C++/Java           Solución GoF        En Python
---------------------------------------------------------------------------
Necesitas interface formal  -->  Pattern usando     --> Duck typing (no hay
para polimorfismo               polimorfismo          que declararla)
Funciones no son valores    -->  Strategy / Command --> Funciones de 1ra clase
Clases pesadas de instanciar-->  Factory            --> Factory = simple función
No hay metaclasses / dinámica-> Patrones elaborados --> Decoradores, metaclases,
                                                   --> __getattr__, protocols
```

**Principio clave:** antes de implementar un patrón, pregúntate *"¿tiene Python ya un idioma nativo para esto?"*. A menudo la respuesta es **sí**.

---

## 2. Patrones Creacionales

Se ocupan de **cómo se crean los objetos**, desacoplando la creación del uso.

### 2.1 Factory Method

**Problema:** el cliente no debe saber *qué clase concreta* está creando. La decisión se centraliza en una **fábrica**.

```
               Cliente
                  |
                  |  pide tipo: "csv"
                  v
           +------------------+
           |   serializer()   |  <- Fábrica (función)
           +------------------+
                  |
         +--------+--------+
         |                 |
         v                 v
   +----------+      +----------+
   | CSVWriter|      |JSONWriter|
   +----------+      +----------+
        \               /
         \             /
          v           v
     comparten: escriben(data)
```

In [ ]:
from abc import ABC, abstractmethod


class Exporter(ABC):
    """Producto abstracto: la 'interfaz' común."""
    @abstractmethod
    def export(self, data: dict) -> str: ...


class CsvExporter(Exporter):
    def export(self, data: dict) -> str:
        return "\n".join(f"{k},{v}" for k, v in data.items())


class JsonExporter(Exporter):
    def export(self, data: dict) -> str:
        import json
        return json.dumps(data, ensure_ascii=False)


def make_exporter(fmt: str) -> Exporter:
    """Fábrica: decide qué producto concreto devolver."""
    factories = {"csv": CsvExporter, "json": JsonExporter}
    try:
        return factories[fmt.lower()]()
    except KeyError:
        raise ValueError(f"Formato no soportado: {fmt}")


data = {"nombre": "Ana", "edad": 30}
print(make_exporter("csv").export(data))
print(make_exporter("json").export(data))

**Observación:** en Python la fábrica suele ser una **función ordinaria** (no una subclase). Y gracias al `dict` de fabricación, agregar un formato nuevo solo requiere añadir una entrada.

### 2.2 Abstract Factory

**Problema:** necesitas crear **familias de objetos relacionados** garantizando consistencia (ej. todos los componentes de una misma UI). Futuro: permite cambiar la familia completa.

```
               Cliente
                  |
       Elige familia: "claro" | "oscuro"
                  v
   +-------------------------------+---+
   | abstract_factory(tema)         |
   +-------------------------------+---+
        |                  |
        v                  v
   Familia Claro       Familia Oscuro
   boton()  ventana()  boton()  ventana()
```

In [ ]:
class Boton(ABC):
    @abstractmethod
    def pintar(self) -> str: ...

class Ventana(ABC):
    @abstractmethod
    def fondo(self) -> str: ...


class BotonClaro(Boton):
    def pintar(self) -> str:
        return "boton blanco"

class VentanaClara(Ventana):
    def fondo(self) -> str:
        return "blanco"

class BotonOscuro(Boton):
    def pintar(self) -> str:
        return "boton negro"

class VentanaOscura(Ventana):
    def fondo(self) -> str:
        return "negro"


def tema_factory(tema: str) -> dict[str, type[Boton] | type[Ventana]]:
    """Crea una FAMILIA consistente de objetos relacionados."""
    familias = {
        "claro": {"boton": BotonClaro, "ventana": VentanaClara},
        "oscuro": {"boton": BotonOscuro, "ventana": VentanaOscura},
    }
    return familias[tema]


familia = tema_factory("oscuro")
b = familia["boton"]()
v = familia["ventana"]()
print(f"Tema oscuro => {v.fondo()} con {b.pintar()}")

familia = tema_factory("claro")
print(f"Tema claro => {familia['ventana']().fondo()} con {familia['boton']().pintar()}")

### 2.3 Singleton (y sus críticas en Python)

**Problema:** garantizar que una clase tenga **una sola instancia** global (ej. configuración, conexión).

**En Python hay 3 formas idiomáticas:**

1. **Module Singleton** (favorita): un módulo solo se importa una vez; un objeto a nivel de módulo ya es un singleton.
2. **`__new__`**: controlar la creación de instancia.
3. **Clase Singleton clásica** (a evitar): pesada y poco pitónica.

In [ ]:
# Modo 1: Module Singleton (idiomático y recomendado)
class Config:  # clase normal, sin magia
    def __init__(self) -> None:
        self.db_url = "sqlite:///:memory:"

config = Config()  # instancia UNICA a nivel de modulo


# Modo 2: Singleton con __new__ (cuando se quiere fuerza)
class SoloUno:
    _instancia = None

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
        return cls._instancia


a = SoloUno()
b = SoloUno()
print("Misma instancia?", a is b)
print("Module singleton:", config.db_url)

> **Crítica profesional:** en Python el *module singleton* casi siempre es suficiente y evita la complejidad del patrón. El anti-patrón clásico (con `cls._instancia`) se reserva para casos puntuales (metaclases, pruebas). Además, muchos usos de Singleton son considerados un *code smell* que esconde un **estado global acoplado**.

### 2.4 Builder

**Problema:** construir un objeto complejo con muchos pasos (o parámetros opcionales) se vuelve ilegible con un constructor gigante.

```
constructor()  -> breabiar()  -> procedimientos
     Builder
         |
     set_harina()
     set_levadura()
     set_tiempo()   (pasos encadenados)
         |
         v
      build() -> Pan
```

In [ ]:
class Pan:
    def __init__(self) -> None:
        self.tipo = None
        self.ingredientes: list[str] = []

    def __repr__(self) -> str:
        return f"Pan({self.tipo}: {', '.join(self.ingredientes)})"


class PanBuilder:
    def __init__(self) -> None:
        self._pan = Pan()

    def de_tipo(self, tipo: str) -> "PanBuilder":
        self._pan.tipo = tipo
        return self  # permitir encadenar

    def con_ingrediente(self, ing: str) -> "PanBuilder":
        self._pan.ingredientes.append(ing)
        return self

    def build(self) -> Pan:
        return self._pan


pan = (PanBuilder()
       .de_tipo("integral")
       .con_ingrediente("harina")
       .con_ingrediente("sal")
       .build())
print(pan)

> **En Python moderno** para datos simples se prefiere `@dataclass(frozen=True)` con valores por defecto en vez de un Builder. El Builder gana cuando hay **lógica de construcción** (validaciones, orden de pasos, dependencias), no solo seteo de atributos.

---

## 3. Patrones Estructurales

Definen **cómo se componen** clases y objetos para formar estructuras más grandes.

### 3.1 Adapter

**Problema:** dos interfaces incompatibles; necesitas que una funcione con la otra **sin modificar** el código original.

```
  Cliente quiere: .cargar()
          |
          v
     Adapter (funcion/objeto)
          |
          v
  API externa: .fetch_rows()
```

In [ ]:
# API externa (no la podemos tocar)
class ApiExterna:
    def fetch_rows(self) -> list[tuple]:
        return [("a", 1), ("b", 2)]


# El resto de la app espera: .cargar() -> list[dict]
# Adaptador con una FUNCION (idioma python)
def adaptar(api: ApiExterna) -> list[dict]:
    return [{"letra": letra, "num": num} for letra, num in api.fetch_rows()]


datos = adaptar(ApiExterna())
print(datos)

**En Python** un adapter a menudo es una simple **función que convierte**, o una propiedad/`__getattr__`, en lugar de una clase con roles formales.

### 3.2 Facade

**Problema:** un subsistema complejo con muchas partes; quieres exponer una **interfaz simplificada**.

```
      Cliente
         |
         v
   +-------------+
   |  Facade     |  cocinar_pizza()
   +-------------+
      |   |   |
      v   v   v
   Masa  Salsa  Horno   <- subsistema
```

In [ ]:
class Masa:
    def preparar(self) -> str:
        return "masa preparada"

class Salsa:
    def aplicar(self) -> str:
        return "salsa aplicada"

class Horno:
    def hornear(self) -> str:
        return "horneando 10 min"


class PizzaFacade:
    """Enmascara los pasos internos del subsistema."""
    def __init__(self) -> None:
        self.masa, self.salsa, self.horno = Masa(), Salsa(), Horno()

    def pizza_lista(self) -> list[str]:
        return [
            self.masa.preparar(),
            self.salsa.aplicar(),
            self.horno.hornear(),
            "pizza servida",
        ]


print(PizzaFacade().pizza_lista())

### 3.3 Proxy

**Problema:** necesitas **controlar el acceso** a un objeto costoso o sensible (lazy loading, permiso, caching) sin que el cliente lo note.

```
   Cliente  --->  Proxy (control)  --->  ObjetoReal
                     |
              + lazy: solo crea al usar
              + log / permisos / cache
```

In [ ]:
import time


class ServicioReal:
    def consultar(self, clave: str) -> str:
        time.sleep(0.2)  # costoso
        return f"valores[{clave}]"


class ProxyCache:
    """Mismo interface, agrega cache y log."""
    def __init__(self, real: ServicioReal) -> None:
        self._real = real
        self._cache: dict[str, str] = {}

    def consultar(self, clave: str) -> str:
        if clave not in self._cache:
            print(f"[proxy] cargando {clave}...")
            self._cache[clave] = self._real.consultar(clave)
        return self._cache[clave]


servicio = ProxyCache(ServicioReal())
print(servicio.consultar("x"))   # lento (primera vez)
print(servicio.consultar("x"))   # rapido (cache)

### 3.4 Decorator (patrón POO: envolver de forma flexible)

**Ojo:** aquí hablamos del **patrón Decorator** (agregar comportamiento envolviendo objetos), **no** del decorador de sintaxis `@`. Ambos comparten la *idea* de envolver.

```
   Componente base
        |
   +----+----+
   |         |
  Concreto   Decorador (envuelve)
                 |
        +--------+--------+
        |                 |
   Decorador A        Decorador B
    (comision)          (log)
```

In [ ]:
class Bebida(ABC):
    @abstractmethod
    def costo(self) -> int: ...

    @abstractmethod
    def descripcion(self) -> str: ...


class Cafe(Bebida):
    def costo(self) -> int:
        return 10

    def descripcion(self) -> str:
        return "cafe"


class DecoradorBebida(Bebida, ABC):
    def __init__(self, bebida: Bebida) -> None:
        self._bebida = bebida


class ConLeche(DecoradorBebida):
    def costo(self) -> int:
        return self._bebida.costo() + 3

    def descripcion(self) -> str:
        return self._bebida.descripcion() + " + leche"


class ConCrema(DecoradorBebida):
    def costo(self) -> int:
        return self._bebida.costo() + 5

    def descripcion(self) -> str:
        return self._bebida.descripcion() + " + crema"


bebida = ConCrema(ConLeche(Cafe()))  # envuelve de forma flexible
print(bebida.descripcion(), "->", bebida.costo())

> En Python, cuando el envoltorio es **simple y horizontal**, el decorador de sintaxis `@` o los *context managers* suelen ser más pitónicos que la cadena de objetos del patrón clásico.

---

## 4. Patrones Comportamentales

Definen **cómo colaboran y se comunican** los objetos.

### 4.1 Strategy

**Problema:** tienes varios algoritmos para una misma tarea y quieres intercambiarlos en tiempo de ejecución.

```
   Contexto
     |
     +--> estrategia (cualquier callable)
             |
         +---+----+-----+
         |    |    |     |
        min  max  mean  custom
```

In [ ]:
# Las estrategias son funciones ordinarias (primera clase)
def estrategia_min(numeros: list[int]) -> int:
    return min(numeros)


def estrategia_max(numeros: list[int]) -> int:
    return max(numeros)


def estrategia_media(numeros: list[int]) -> float:
    return sum(numeros) / len(numeros)


class Calculadora:
    def __init__(self, estrategia) -> None:
        self.estrategia = estrategia  # inyectada

    def calcular(self, numeros: list[int]):
        return self.estrategia(numeros)


calc = Calculadora(estrategia_media)
print("media:", calc.calcular([1, 2, 3, 4, 5]))
calc.estrategia = estrategia_max  # intercambiable en runtime
print("max:", calc.calcular([1, 2, 3, 4, 5]))

### 4.2 Observer

**Problema:** un objeto (sujeto) debe **notificar cambios** a múltiples objetos (observadores) sin acoplarse a ellos.

```
        Sujeto
     +--+--+--+
     |  |  |  |      observadores (callables)
     v  v  v  v
   obs1 obs2 obs3
       ^
    estado cambia -> notifica a todos
```

In [ ]:
class Sujeto:
    def __init__(self) -> None:
        self._observadores: list[callable] = []
        self._estado = 0

    def suscribir(self, obs: callable) -> None:
        self._observadores.append(obs)

    def cambiar(self, valor: int) -> None:
        self._estado = valor
        # notificar a todos (funciones = observadores)
        for obs in self._observadores:
            obs(self._estado)


def alertar(val: int) -> None:
    print(f"[alerta] valor es {val}")


def registrar(val: int) -> None:
    print(f"[log] se cambio a {val}")


s = Sujeto()
s.suscribir(alertar)
s.suscribir(registrar)
s.cambiar(42)

> En Python los observadores suelen ser **funciones/callables** (o *callbacks*), no clases con un método `update`. Esto elimina mucho ceremonial del patrón clásico.

### 4.3 Command

**Problema:** encapsular una **acción** (con sus parámetros) para poder pasarla, deshacerla o encolarla.

```
   Invoker (cola/UI)
        |
        v
   Command: envoltura de accion
        |
     exec() -> Receptor
     undo() -> Receptor
```

In [ ]:
class EditorTexto:
    """Receptor: el objeto que ejecuta las acciones."""
    def __init__(self) -> None:
        self.texto = ""

    def escribir(self, txt: str) -> None:
        self.texto += txt

    def borrar(self, n: int) -> None:
        self.texto = self.texto[:-n]


class Comando(ABC):
    @abstractmethod
    def ejecutar(self) -> None: ...

    @abstractmethod
    def deshacer(self) -> None: ...


class EscribirComando(Comando):
    def __init__(self, editor: EditorTexto, txt: str) -> None:
        self._editor, self._txt = editor, txt

    def ejecutar(self) -> None:
        self._editor.escribir(self._txt)

    def deshacer(self) -> None:
        self._editor.borrar(len(self._txt))


# Invoker: encola y permite deshacer
editor = EditorTexto()
historial: list[Comando] = []

cmd = EscribirComando(editor, "hola ")
cmd.ejecutar(); historial.append(cmd)
cmd = EscribirComando(editor, "mundo")
cmd.ejecutar(); historial.append(cmd)
print("texto:", editor.texto)

historial.pop().deshacer()
print("tras deshacer:", editor.texto)

> En Python, cuando la acción es simple, una **closure/función** ya la encapsula. El Command en objeto aporta valor cuando necesitas **deshacer** o **serializar** la acción (ej. logs de eventos).

### 4.4 Template Method

**Problema:** defines el **esqueleto** de un algoritmo y dejas que las subclases redefinan ciertos pasos.

```
   ClaseBase.template()
       paso1()  (fijo)
       paso2()  (abstracto -> subclase)
       paso3()  (fijo)
```

In [ ]:
class AnalizadorDatos(ABC):
    """Template: esqueleto del pipeline."""
    def ejecutar(self) -> list[str]:
        pasos = []
        pasos.append("1. cargar")
        pasos.append(self._transformar())  # paso variable
        pasos.append("3. reportar")
        return pasos

    @abstractmethod
    def _transformar(self) -> str: ...


class AnalizadorCSV(AnalizadorDatos):
    def _transformar(self) -> str:
        return "2. limpiar CSV"


class AnalizadorJSON(AnalizadorDatos):
    def _transformar(self) -> str:
        return "2. parsear JSON"


for analizador in (AnalizadorCSV(), AnalizadorJSON()):
    print(analizador.ejecutar())

---

## 5. Patrones que en Python se vuelven "nativos"

La magia de Python es que muchas soluciones ya están **incrustadas en el lenguaje** y evitan implementar el patrón explícitamente.

In [ ]:
# 1) Command -> simple llamada a metodo / funcion
acciones = []
acciones.append(lambda: print("accion A"))
acciones.append(lambda: print("accion B"))
for a in acciones:
    a()

# 2) Strategy -> funcional (funciones/lambdas como deposito de estrategias)
ordenadores = {"asc": sorted, "desc": lambda xs: sorted(xs, reverse=True)}
print(ordenadores["desc"]([3, 1, 2]))

# 3) Singleton -> module singleton (ya visto)
# 4) Builder -> @dataclass
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Configuracion:
    host: str = "localhost"
    puerto: int = 5432
    debug: bool = field(default=False, kw_only=True)

cfg = Configuracion(host="db", puerto=1234, debug=True)
print(cfg)

# 5) Context managers -> cobertura segura (try/finally idomatico)
class Conexion:
    def __enter__(self):
        print("abrir conexion")
        return self
    def __exit__(self, *exc):
        print("cerrar conexion")

with Conexion() as c:
    print("trabajando...")

### Resumen de "idiomas nativos"

| Patrón GoF | Reemplazo idiomático en Python |
|------------|--------------------------------|
| Command | Funciones / closures / `functools.partial` |
| Strategy | Funciones de primera clase, lambdas, `functools` |
| Singleton | Module singleton, `functools.lru_cache`, metaclases |
| Builder | `@dataclass` con valores por defecto, `attrs`, `pydantic` |
| Observer | Event system con callbacks, `asyncio`, señales (blinker) |
| Commercial | No aplica / siempre verificar si el problema lo merece |

---

## 6. Anti-patrones a evitar

Igual de importante que conocer patrones es **no abusar de ellos**.

| Anti-patrón | Descripción | Señal de alerta | Solución en Python |
|-------------|-------------|-----------------|--------------------|
| **God Object** | Una clase que hace TODO | Clases de +1000 líneas, muchos métodos no relacionados | Separar responsabilidades, composición |
| **Dioses de utilidades** | Módulos `utils.py` gigantes con funciones sueltas | `from utils import *` | Agrupar por dominio, module singletons, namespacing |
| **Circular dependency** | Módulo A importa B y B importa A | `ImportError: cannot import name` | Inyección de dependencias, interfaces/hooks, ajustar imports |
| **Over-engineering** | Aplicar patrones innecesarios | Distancia de indirección alta, abstracciones sin uso | YAGNI (You Ain't Gonna Need It) |
| **Premature abstraction** | Abstraer antes de tener 2-3 usos reales | Factorías/estrategias para un único caso | Refactorizar **después** de ver duplicación real |

**Regla de oro:** el patrón correcto es el que **reduce** la complejidad de tu caso, no el que crea la mayor cantidad de clases.

---

## 7. Tabla de referencia rápida

| Patrón | Problema que resuelve | Implementación en Python |
|--------|----------------------|--------------------------|
| Factory Method | Crear objetos sin acoplar al tipo | Función con `dict` de fábricas |
| Abstract Factory | Crear familias consistentes | Función que retorna dict/clase de familia |
| Singleton | Una sola instancia global | Module singleton / `__new__` |
| Builder | Construcción compleja paso a paso | Clase con métodos encadenados (`self`) o `@dataclass` |
| Adapter | Interfaces incompatibles | Función convertidora / `__getattr__` |
| Facade | Interfaz simple ante subsistema | Clase que envuelve varios objetos |
| Proxy | Control/cache/acceso a objeto | Clase con mismo interface + lógica |
| Decorator | Agregar comportamiento envolviendo | Decoradores `@`, context managers, objetos que envuelven |
| Strategy | Algoritmos intercambiables | Funciones/`functools` como estrategias |
| Observer | Notificar a muchos sin acoplar | Callbacks/`list` de funciones, `asyncio` |
| Command | Encapsular acción (deshacer/cola) | Funciones/closures, o clases con `ejecutar/deshacer` |
| Template Method | Esqueleto con pasos variables | Clase ABC con método concreto + abstracto |

---

## 8. Ejercicios

### Ejercicio 1 (Guía) — Factory Method

Crea una fábrica `make_formato` que devuelva `FormatoTexto` distintos según el tipo (`upper`, `lower`, `title`). Cada producto expone `aplicar(texto: str) -> str`.

In [ ]:
from abc import ABC, abstractmethod

class FormatoTexto(ABC):
    @abstractmethod
    def aplicar(self, texto: str) -> str: ...

class FormatoUpper(FormatoTexto):
    def aplicar(self, texto: str) -> str:
        return texto.upper()

class FormatoLower(FormatoTexto):
    def aplicar(self, texto: str) -> str:
        return texto.lower()

class FormatoTitle(FormatoTexto):
    def aplicar(self, texto: str) -> str:
        return texto.title()

def make_formato(tipo: str) -> FormatoTexto:
    f = {"upper": FormatoUpper, "lower": FormatoLower, "title": FormatoTitle}
    return f[tipo]()

texto = "hola mundo python"
for t in ("upper", "lower", "title"):
    print(make_formato(t).aplicar(texto))

### Ejercicio 2 (Guía) — Observer con callbacks

Implementa un sistema de **notificaciones** donde un `Sensor` notifica a observadores cuando cambia la temperatura. Los observadores son funciones.

In [ ]:
class Sensor:
    def __init__(self) -> None:
        self._obs: list[callable] = []

    def suscribir(self, obs: callable) -> None:
        self._obs.append(obs)

    def medir(self, temp: float) -> None:
        for obs in self._obs:
            obs(temp)

def alerta_alta(temp: float) -> None:
    if temp > 40:
        print(f"PELIGRO: {temp}°C!")

def registro(temp: float) -> None:
    print(f"log: temperatura {temp}")

s = Sensor()
s.suscribir(alerta_alta)
s.suscribir(registro)
s.medir(20)
s.medir(45)

### Ejercicio 3 (Guía) — Strategy con funciones

Crea una `CalculadoraDescuento` que reciba una **estrategia** de descuento (función) y la aplique a un precio. Cambia de estrategia en runtime.

In [ ]:
def descuento_nada(precio: float) -> float:
    return precio

def descuento_10(precio: float) -> float:
    return precio * 0.9

def descuento_fijo(rebaja: float):
    return lambda precio: max(0, precio - rebaja)

class CalculadoraDescuento:
    def __init__(self, estrategia) -> None:
        self.estrategia = estrategia

    def total(self, precio: float) -> float:
        return self.estrategia(precio)

calc = CalculadoraDescuento(descuento_10)
print("con 10%:", calc.total(100))
calc.estrategia = descuento_fijo(25)
print("con rebaja 25:", calc.total(100))

### Ejercicio 4 (Independiente) — Sistema de Notificaciones completo

Diseña un sistema de notificaciones que combine **Observer + Strategy + Factory**, usando idiomas de Python.

**Requisitos:**

1. **Factory**: `make_canal(tipo)` devuelve un canal (`Email`, `SMS`, `Push`).
2. **Strategy**: cada canal tiene una estrategia de *formato* del mensaje (simple / markdown).
3. **Observer**: un `Notificador` central recoge canales y, al emitir un evento, notifica a todos.
4. Uso de **funciones** como estrategias y **callbacks** como observadores.

Ten en cuenta: no sobre-ingeniar (YAGNI); añade solo la abstracción que aporte valor.

**Pista:** empieza por los canales (función `make_canal`), luego las estrategias de formato (funciones `simple`/`markdown`), y termina con el `Notificador` que los agrupa.

In [ ]:
# --- Tu solucion aqui ---
# 1. Canales (Factory)
# 2. Estrategias de formato (funciones)
# 3. Notificador (Observer que agrupa canales)


# === Puntos de control ===
# n = Notificador()
# n.agregar(make_canal("email"), estrategia=formato_markdown)
# n.agregar(make_canal("sms"), estrategia=formato_simple)
# n.notificar_todos("Hola", usuario="Ana")
# Esperado: cada canal recibe el evento y lo formatea a su manera.

### Solución de referencia (Ejercicio 4)

Esta es una **posible** solución, no la única correcta.

In [ ]:
from abc import ABC, abstractmethod

# Estrategias de formato (funciones de primera clase)
def formato_simple(mensaje: str, usuario: str) -> str:
    return f"[para {usuario}] {mensaje}"

def formato_markdown(mensaje: str, usuario: str) -> str:
    return f"**Hola {usuario}:** *{mensaje}*"

# Canales (productos de la fábrica)
class Canal(ABC):
    def __init__(self) -> None:
        self.estrategia = formato_simple

    def enviar(self, mensaje: str, usuario: str) -> None:
        print(f"{self.tipo}-> {self.estrategia(mensaje, usuario)}")

    @property
    @abstractmethod
    def tipo(self) -> str: ...

class CanalEmail(Canal):
    @property
    def tipo(self) -> str:
        return "email"

class CanalSms(Canal):
    @property
    def tipo(self) -> str:
        return "sms"

class CanalPush(Canal):
    @property
    def tipo(self) -> str:
        return "push"

# Factory de canales
def make_canal(tipo: str) -> Canal:
    canales = {"email": CanalEmail, "sms": CanalSms, "push": CanalPush}
    return canales[tipo.lower()]()

# Observer central (agrupa canales)
class Notificador:
    def __init__(self) -> None:
        self._canales: list[Canal] = []

    def agregar(self, canal: Canal, estrategia=formato_simple) -> None:
        canal.estrategia = estrategia  # Strategy inyectada
        self._canales.append(canal)

    def notificar_todos(self, mensaje: str, usuario: str) -> None:
        for canal in self._canales:  # notifica a todos (Observer)
            canal.enviar(mensaje, usuario)


n = Notificador()
n.agregar(make_canal("email"), estrategia=formato_markdown)
n.agregar(make_canal("sms"), estrategia=formato_simple)
n.notificar_todos("Tienes una nueva tarea", usuario="Ana")

---

## 9. Resumen

- Los **23 patrones GoF** son recetas probadas; en Python muchos se **simplifican** con duck typing, funciones de primera clase, decoradores y dataclasses.
- **Creacionales**: Factory Method (función fábrica), Abstract Factory (familias), Singleton (module singleton/`__new__`), Builder (encadenado/`@dataclass`).
- **Estructurales**: Adapter (conversión), Facade (interfaz simple), Proxy (control/cache), Decorator (envolver).
- **Comportamentales**: Strategy (funciones), Observer (callbacks), Command (encapsular acción), Template Method (herencia).
- Los idiomas nativos de Python (funciones, closures, `with`, `@dataclass`) **sustituyen** con frecuencia al patrón explícito.
- Evita anti-patrones: God Object, dioses de utilidades, dependencias circulares, sobre-ingeniería y abstracción prematura (**YAGNI**).

> **Conclusión clave:** el valor de conocer patrones está en **reconocer el problema** y elegir el idioma de Python que lo resuelva con la **menor complejidad** posible.